# Local background and detector threshold study
This study evaluates the current integrated detector and classifier. It chooses a confidence threshold using only selection-validation and local training images. The historical team test is shown afterwards and is not an independent test.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess, tempfile, json

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path(tempfile.mkdtemp())/'project'
subprocess.run(['git','clone','--quiet','--depth','1','--branch','project',
                'https://github.com/salehdev3/Capstone_5.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
sys.path.insert(0,str(PROJECT/'runtime'))
for module in ('live','deployment','combined','candidate','guided','crop_model','crop_sam',
               'bf16_encoder','http_service','runtime','common'):
    sys.modules.pop(module,None)
from common import read, save, pairs, sha
from live import initialize, execute, team_test_cases, RUN
from deployment import GuidedPipeline
import numpy as np
import pandas as pd
from IPython.display import display, Image

OUT = ROOT/'phase2_results/local_threshold_study'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT.mkdir(parents=True)
pipeline, decision = initialize(ROOT)
rows = read(ROOT/'detection_manifest.json')
selection = [r for r in rows if r['split']=='val_select']
local_train = [r for r in rows if r['split']=='train' and r.get('source')=='team_collected']
if not selection or not local_train:
    raise ValueError('Selection validation and team-collected training images are required.')
print('Selection validation:',len(selection),'| local train:',len(local_train))
print('CPU inference is running; progress appears every 50 images.')


In [ ]:
# Cache the actual integrated detector and species output with masks disabled.
records=[]
for i,row in enumerate(selection+local_train,1):
    path=ROOT/'images'/(row['id']+'.jpg')
    if not path.is_file(): raise FileNotFoundError(path)
    frame=pipeline.process_frame(path,row,with_masks=False)
    records.append({'id':row['id'],'split':row['split'],'source':row.get('source'),
                    'animal':bool(row['animal']),'gt':[b['xyxy'] for b in row['boxes']],
                    'objects':frame['objects']})
    if i%50==0 or i==len(selection)+len(local_train):
        print('Processed',i,'/',len(selection)+len(local_train),flush=True)
save(OUT/'selection_predictions.json',records)

def evaluate(rows, threshold):
    animal_count=empty_count=animal_found=empty_removed=tp=fp=fn=0
    for row in rows:
        preds=[o['box'] for o in row['objects'] if o['detector_confidence']>=threshold]
        matches=len(pairs(row['gt'],preds))
        tp+=matches
        fp+=len(preds)-matches
        fn+=len(row['gt'])-matches
        if row['animal']:
            animal_count+=1
            animal_found+=bool(preds)
        else:
            empty_count+=1
            empty_removed+=not preds
    return {'animal_recall':animal_found/animal_count if animal_count else None,
            'empty_removal':empty_removed/empty_count if empty_count else None,
            'box_precision':tp/(tp+fp) if tp+fp else 0,
            'box_recall':tp/(tp+fn) if tp+fn else 0,
            'tp':tp,'fp':fp,'fn':fn,'animal_frames':animal_count,'empty_frames':empty_count}

val_records=[r for r in records if r['split']=='val_select']
local_records=[r for r in records if r['split']=='train']
baseline_conf=float(pipeline.config['detector']['conf'])
thresholds=sorted(set([baseline_conf,.025,.03,.035,.04,.045,.05,.075,.10,.15,.20,.25,.30]))
thresholds=[v for v in thresholds if v>=baseline_conf]
sweep=[]
for threshold in thresholds:
    val=evaluate(val_records,threshold)
    local=evaluate(local_records,threshold)
    sweep.append({'threshold':threshold,**{'val_'+k:v for k,v in val.items()},
                  **{'local_train_'+k:v for k,v in local.items()}})
pd.DataFrame(sweep).to_csv(OUT/'threshold_sweep.csv',index=False)
display(pd.DataFrame(sweep))
# Retain the original validation criteria. Prefer fewer local training empty-frame
# false positives, then better validation box precision and the lower threshold.
eligible=[r for r in sweep if r['val_animal_recall']>=.99 and r['val_empty_removal']>=.20]
if not eligible:
    save(OUT/'selection.json',{'selected':None,'reason':'No candidate satisfies validation acceptance.'})
    raise ValueError('No threshold meets the validation recall and empty-frame criteria.')
chosen=max(eligible,key=lambda r:(r['local_train_empty_removal'] if r['local_train_empty_removal'] is not None else -1,
                                  r['val_box_precision'],-r['threshold']))
save(OUT/'selection.json',{'selected':chosen,'parent_policy_sha256':sha(ROOT/RUN/'policy.json'),
                            'selection_scope':'val_select and team-collected train only',
                            'historical_team_test_independent':False})
print('Selected confidence:',chosen['threshold'])


In [ ]:
# Evaluate the selected threshold on the historical team images and save visuals.
# This produces experimental evidence and does not change the existing demo policy.
import copy
policy=copy.deepcopy(read(ROOT/RUN/'policy.json'))
policy['detector']['conf']=chosen['threshold']
policy['parent_policy_sha256']=sha(ROOT/RUN/'policy.json')
save(OUT/'candidate_policy.json',policy)
candidate=GuidedPipeline(ROOT,Path(tempfile.mkdtemp(prefix='threshold_study_')),OUT/'candidate_policy.json')
folder,result=execute(candidate,team_test_cases(ROOT),OUT/'team_test')
display(pd.DataFrame([{'id':f['id'],'count':f['count'],'seconds':f['timing']['total'],
                       'species':', '.join(o['species'] for o in f['objects'])}
                      for f in result['frames']]))
for path in sorted(folder.glob('frame_*.jpg')):
    display(Image(filename=str(path),width=480))
print('Study folder:',OUT)
print('Team test output:',folder)
